## Reading Bronze.Airports Delta Table

In [0]:
Airports_bronze_path = "s3://travel-analytics-bronze/delta/bronze/airports/"
Airports_bronze_df = spark.read.format("delta").load(Airports_bronze_path)

## Silver Transformations

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, trim, upper, to_date, coalesce, lit, when ,to_timestamp

# =============================================================
# STEP 0: CONFIGURATION & SETUP
# =============================================================
table_name = "airports"
airports_silver_path = f"s3://travel-analytics-bronze/delta/silver/{table_name}/"

# =============================================================
# STEP 1: DATA TYPE CASTING & Parsing
# =============================================================
print("\n STEP 1: Casting Data Types...")

airports_step_1_df = (
    Airports_bronze_df
    
    # Numeric columns
    .withColumn("airport_id", col("airport_id").cast("int"))
    .withColumn("latitude", col("latitude").cast("double"))
    .withColumn("longitude", col("longitude").cast("double"))
    
    # String columns: trim and uppercase
    .withColumn("city", trim(col("city")))
    .withColumn("airport_name", trim(col("airport_name")))
    .withColumn("airport_type", trim(upper(col("airport_type"))))
    
    # ========== CDC updated timestamp ==========
    .withColumn("updated_at", to_timestamp(col("_ab_cdc_updated_at"))))

# =============================================================
# STEP 2: Cleaning, Standardizing Strings and Business Logic
# =============================================================
print("\n STEP 2: Cleaning, Standardizing String Columns...")

airports_step_2_df = (
    airports_step_1_df
    
    # Fill missing city or airport_name with UNKNOWN
    .withColumn("city", coalesce(col("city"), lit("UNKNOWN")))
    .withColumn("airport_name", coalesce(col("airport_name"), lit("UNKNOWN")))
    
    # Uppercase standardization for airport_type
    .withColumn("airport_type", coalesce(col("airport_type"), lit("UNKNOWN")))
)

# =============================================================
# STEP 3: Deduplicating Using Composite PK("airport_id")
#          Dropping unwanted Airbyte metadata columns
# =============================================================
print("\n STEP 3: Deduplicating and Dropping unwanted Airbyte metadata columns...")

airbyte_columns_to_drop = [
    "_airbyte_ab_id",
    "_airbyte_emitted_at",
    "_ab_cdc_lsn",
    "_airbyte_additional_properties",
    "_ab_cdc_deleted_at",
    "_ab_cdc_updated_at"
]

airports_step_3_df = (
    airports_step_2_df
    
    # Drop duplicates based on airport_id
    .dropDuplicates(["airport_id"])
    
    # Drop Airbyte metadata
    .drop(*airbyte_columns_to_drop)
)


 STEP 1: Casting Data Types...

 STEP 2: Cleaning, Standardizing String Columns...

 STEP 3: Deduplicating and Dropping unwanted Airbyte metadata columns...


In [0]:
# =============================================================
# STEP 4: COLUMN RENAMING (Business-Friendly Names)
# =============================================================
print("\nSTEP 4: Renaming Columns to Business Standards...")

rename_map = {
    "airport_id": "Airport_Id",
    "city": "City",
    "airport_name": "Airport_Name",
    "airport_type": "Airport_Type",
    "latitude": "Latitude",
    "longitude": "Longitude",
    "updated_at": "Updated_At"
}

airports_silver_df = airports_step_3_df.select(
    [col(c).alias(rename_map.get(c, c)) for c in airports_step_3_df.columns]
)


STEP 4: Renaming Columns to Business Standards...


In [0]:
airports_silver_df.display()

City,Latitude,Longitude,Airport_Id,Airport_Name,Airport_Type,Updated_At
Yakutsk,62.0933,129.77,1,Yakutsk Airport,DOMESTIC,2025-12-12T01:04:52.921Z
Mirnyj,62.5347,113.961,2,Mirny Airport,DOMESTIC,2025-12-12T01:04:52.921Z
Khabarovsk,48.528,135.188,3,Khabarovsk-Novy Airport,DOMESTIC,2025-12-12T01:04:52.921Z
Petropavlovsk,53.1679,158.453,4,Yelizovo Airport,DOMESTIC,2025-12-12T01:04:52.921Z
Yuzhno-Sakhalinsk,46.8887,142.718,5,Yuzhno-Sakhalinsk Airport,DOMESTIC,2025-12-12T01:04:52.921Z
Vladivostok,43.399,132.148,6,Vladivostok International Airport,INTERNATIONAL,2025-12-12T01:04:52.921Z
St. Petersburg,59.8003,30.2625,7,Pulkovo Airport,DOMESTIC,2025-12-12T01:04:52.921Z
Kaliningrad,54.89,20.5926,8,Khrabrovo Airport,DOMESTIC,2025-12-12T01:04:52.921Z
Kemorovo,55.2701,86.1072,9,Kemerovo Airport,DOMESTIC,2025-12-12T01:04:52.921Z
Chelyabinsk,55.3058,61.5033,10,Chelyabinsk Balandino Airport,DOMESTIC,2025-12-12T01:04:52.921Z


In [0]:
# =============================================================
# STEP 5: Write to Silver Layer
# =============================================================
airports_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(airports_silver_path)

print(f"Airports table saved to Silver layer at {airports_silver_path}")

Airports table saved to Silver layer at s3://travel-analytics-bronze/delta/silver/airports/
